# Industrial-Safety Datasets and Dashboard

"Industrial_Safety" data cleaing, preprocessing, and dashboard visualization.

In [2]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, to_date, lower, trim
from pyspark.sql.functions import sum as spark_sum


%pip install --upgrade pandas
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

%pip install dash plotly

%pip install wordcloud
from wordcloud import WordCloud
import base64
from io import BytesIO
import re

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
'''
Download and load dataset to dataframe
'''

# Start Spark session
spark = SparkSession.builder.appName("Industrial_Safety").getOrCreate()

# Download and save dataset
import requests
path = "https://raw.githubusercontent.com/huqingyuan314/DS5110-25Summer-Project/refs/heads/main/datasets/Industrial%20Safety%20and%20Health%20Analytics%20Database/IHMStefanini_industrial_safety_and_health_database_with_accidents_description.csv"
req = requests.get(path)
with open("industrial_safety_data.csv", "wb") as f:
    f.write(req.content)

# Load CSV with header and inferred schema
df = spark.read.csv("industrial_safety_data.csv", header=True, inferSchema=True)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/01 13:03:31 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.220 instead (on interface en0)
25/08/01 13:03:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/01 13:03:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
'''
Data pre-processing
'''

# Filter out non-date records in the 'Data' column
df = df.filter(col("Data").rlike("^[0-9]{4}-"))

# Convert 'Data' to date type
df = df.withColumn("Data", to_date(col("Data"), "yyyy-MM-dd"))

# Standardize and clean categorical fields
categorical_cols = ["Accident Level", "Genre", "Critical Risk"]
for col_name in categorical_cols:
    df = df.withColumn(col_name, trim(lower(col(col_name))))

# Drop records with missing descriptions (for word cloud quality)
df = df.filter(col("Description").isNotNull())

# Drop records with nulls in other critical columns
critical_cols = ["Accident Level", "Critical Risk", "Genre"]
df = df.dropna(subset=critical_cols)

# Show cleaned schema and sample
df.printSchema()
df.show(5, truncate=False)

root
 |-- Data: date (nullable = true)
 |-- Countries: string (nullable = true)
 |-- Local: string (nullable = true)
 |-- Industry Sector: string (nullable = true)
 |-- Accident Level: string (nullable = true)
 |-- Potential Accident Level: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Employee or Third Party: string (nullable = true)
 |-- Critical Risk: string (nullable = true)
 |-- Description: string (nullable = true)



+----------+----------+--------+---------------+--------------+------------------------+-----+-----------------------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Data      |Countries |Local   |Industry Sector|Accident Level|Potential Accident Level|Genre|Employee or Third Party|Critical Risk      |Description                                                                                                                                                                                                                                

In [5]:
'''
Sanity check
'''

# Count nulls in each column 
print("Count for missing value (nulls) in each column:")
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

# Register for SQL queries
df.createOrReplaceTempView("industrial_safety_data")

Count for missing value (nulls) in each column:
+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+
|Data|Countries|Local|Industry Sector|Accident Level|Potential Accident Level|Genre|Employee or Third Party|Critical Risk|Description|
+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+
|   0|        0|    0|              0|             0|                       0|    0|                      0|            0|          0|
+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+



In [ ]:
'''
=============================================================
Plotly Dash app.
Run this chunk, then go to http://127.0.0.1:8050/ in browser.
=============================================================
'''

%pip install dash
import dash
from dash import dcc, html, dash_table, Input, Output, State
import plotly.express as px
import pandas as pd

# === Load Spark → Pandas dataframe ===
pandas_df = df.toPandas()

# === Parse and clean date ===
pandas_df['DateParsed'] = pd.to_datetime(pandas_df['Data'], errors='coerce')
pandas_df = pandas_df.dropna(subset=['DateParsed'])

# === Add Year, Month, Day columns ===
pandas_df['Year'] = pandas_df['DateParsed'].dt.strftime('%Y')
pandas_df['Month'] = pandas_df['DateParsed'].dt.strftime('%m')
pandas_df['YearMonth'] = pandas_df['DateParsed'].dt.strftime('%Y/%m')
pandas_df['Day'] = pandas_df['DateParsed'].dt.strftime('%d')
pandas_df['FullDate'] = pandas_df['DateParsed'].dt.strftime('%Y/%m/%d')


# ======================
# === Build Dash App ===
# ======================
app = dash.Dash(__name__)
app.title = "Industrial Accident Dashboard"

app.layout = html.Div([
    html.H1("Industrial Safety & Health Dashboard"),

    html.Div([
        html.Label("Year:"),
        dcc.Dropdown(
            id='year-dropdown',
            options=[{"label": y, "value": y} for y in sorted(pandas_df['Year'].unique())],
            value=None,
            placeholder="Select Year"
        ),
    ], style={'width': '200px', 'display': 'inline-block', 'margin-right': '20px'}),

    html.Div([
        html.Label("Month:"),
        dcc.Dropdown(
            id='month-dropdown',
            placeholder="Select Month"
        ),
    ], style={'width': '200px', 'display': 'inline-block', 'margin-right': '20px'}),

    html.Div([
        html.Label("Day:"),
        dcc.Dropdown(
            id='day-dropdown',
            placeholder="Select Day"
        ),
    ], style={'width': '200px', 'display': 'inline-block'}),

    dcc.Graph(id='accidents-chart'),

    html.Div(id='category-pies'),

    html.H3("Accident Cause Word Cloud"),
    html.Div([
    html.Img(id='wordcloud-img', style={'width': '100%', 'maxWidth': '800px'})
    ], style={'textAlign': 'center', 'marginTop': '20px'}),


    html.H3("Accident Records"),
    html.Div([
    dash_table.DataTable(
        id='accident-table',
        columns=[{"name": col, "id": col} for col in pandas_df.columns],
        style_table={'height': '400px', 'overflowY': 'scroll'},  # scrollable
        style_cell={'textAlign': 'left'},
        style_header={'fontWeight': 'bold'},
        page_action='none',  # disable paging
        fixed_rows={'headers': True},  # sticky header
        style_data={'whiteSpace': 'normal', 'height': 'auto'},
    )
])
])

# === Helper Function to Generate Pie Charts (2 * 4 layout) ===
def generate_pie_charts(df, columns):
    pie_charts = []

    for col in columns:
        if col == "Critical Risk":
            # Create Top 10 list
            top10 = df[col].value_counts().nlargest(10).reset_index()
            top10.columns = ["Critical Risk", "Count"]

            rows = [
                html.Tr([html.Td(r["Critical Risk"]), html.Td(r["Count"])])
                for _, r in top10.iterrows()
            ]

            table = html.Table([
                html.Thead(html.Tr([
                    html.Th("Top Critical Risks"), html.Th("Count")
                ])),
                html.Tbody(rows)
            ], style={
                'width': '100%',
                'border-collapse': 'collapse',
                'textAlign': 'left'
            })

            content = html.Div([
                html.H4("Top 10 Critical Risks", style={'margin-bottom': '10px'}),
                table
            ])

            pie_charts.append(
                html.Div(content, style={
                    'width': '24%',
                    'height': '300px',
                    'padding': '10px',
                    'box-sizing': 'border-box',
                    'border': '1px solid #eee',
                    'border-radius': '6px',
                    'background': '#fafafa',
                    'overflowY': 'auto'
                })
            )

        else:
            # Regular pie chart
            counts = df[col].value_counts().reset_index()
            counts.columns = [col, 'Count']
            fig = px.pie(counts, names=col, values='Count', title=f"{col} Distribution")

            pie_charts.append(
                html.Div(
                    dcc.Graph(figure=fig, style={'height': '350px'}),
                    style={
                        'width': '24%',
                        'height': '300px',
                        'padding': '10px',
                        'box-sizing': 'border-box',
                        'overflow': 'hidden'
                    }
                )
            )

    return html.Div(
        children=pie_charts,
        style={
            'display': 'flex',
            'flex-wrap': 'wrap',
            'justify-content': 'space-between',
            'row-gap': '20px',
            'margin-top': '20px'
        }
    )


# === Helper Function to Generate Wordcloud Image ===
def generate_wordcloud_image(text_series):
    # Combine all text into one string
    combined_text = ' '.join(text_series.dropna().astype(str))

    # Remove numbers/punctuation
    combined_text = re.sub(r"[^a-zA-Z\s]", "", combined_text)

    # Create the word cloud
    wc = WordCloud(width=800, height=400, background_color='white',
                   max_words=100, colormap='viridis').generate(combined_text)

    # Convert to base64 for Dash
    buffer = BytesIO()
    wc.to_image().save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode()

    return f"data:image/png;base64,{encoded}"

# === Update month options when year is selected ===
@app.callback(
    Output('month-dropdown', 'options'),
    Output('month-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_months(year):
    if not year:
        return [], None
    months = sorted(pandas_df[pandas_df['Year'] == year]['Month'].unique())
    return [{"label": m, "value": m} for m in months], None

# === Update day options when month is selected ===
@app.callback(
    Output('day-dropdown', 'options'),
    Output('day-dropdown', 'value'),
    Input('year-dropdown', 'value'),
    Input('month-dropdown', 'value')
)
def update_days(year, month):
    if not (year and month):
        return [], None
    days = sorted(pandas_df[
        (pandas_df['Year'] == year) & (pandas_df['Month'] == month)
    ]['Day'].unique())
    return [{"label": d, "value": d} for d in days], None

# === Update chart + table ===
@app.callback(
    Output('accidents-chart', 'figure'),
    Output('accident-table', 'data'),
    Output('category-pies', 'children'),
    Output('wordcloud-img', 'src'),
    Input('year-dropdown', 'value'),
    Input('month-dropdown', 'value'),
    Input('day-dropdown', 'value')
)

def update_output(year, month, day):
    pie_columns = [
        "Countries", "Local", "Industry Sector", "Accident Level",
        "Potential Accident Level", "Genre", "Employee or Third Party", "Critical Risk"
    ]

    if not year:
        df_filtered = pandas_df
        grouped = df_filtered.groupby('Year').size().reset_index(name='Count')
        fig = px.bar(grouped, x='Year', y='Count', title="Yearly Accident Count (All Years)")
        pies = generate_pie_charts(df_filtered, pie_columns)
        wordcloud_src = generate_wordcloud_image(df_filtered["Description"])
        return fig, df_filtered.to_dict('records'), pies, wordcloud_src

    elif year and not month:
        df_filtered = pandas_df[pandas_df['Year'] == year]
        grouped = df_filtered.groupby('Month').size().reset_index(name='Count')
        fig = px.bar(grouped, x='Month', y='Count', title=f"Monthly Accident Count in {year}")
        pies = generate_pie_charts(df_filtered, pie_columns)
        wordcloud_src = generate_wordcloud_image(df_filtered["Description"])
        return fig, df_filtered.to_dict('records'), pies, wordcloud_src

    elif year and month and not day:
        df_filtered = pandas_df[
            (pandas_df['Year'] == year) & (pandas_df['Month'] == month)
        ]
        grouped = df_filtered.groupby('Day').size().reset_index(name='Count')
        fig = px.bar(grouped, x='Day', y='Count', title=f"Daily Accident Count in {year}/{month}")
        pies = generate_pie_charts(df_filtered, pie_columns)
        wordcloud_src = generate_wordcloud_image(df_filtered["Description"])
        return fig, df_filtered.to_dict('records'), pies, wordcloud_src

    elif year and month and day:
        full_date = f"{year}/{month}/{day}"
        df_filtered = pandas_df[pandas_df['FullDate'] == full_date]
        if df_filtered.empty:
            fig = px.bar(title=f"No accidents on {full_date}")
        else:
            grouped = df_filtered.groupby("Industry Sector").size().reset_index(name='Count')
            fig = px.bar(grouped, x="Industry Sector", y="Count", title=f"Accidents on {full_date}")
        wordcloud_src = generate_wordcloud_image(df_filtered["Description"])
        # No pies for full date
        return fig, df_filtered.to_dict('records'), [], wordcloud_src

    return px.bar(title="Invalid Selection"), [], []


# === Program Entry ===
if __name__ == '__main__':
    app.run(debug=True)


Note: you may need to restart the kernel to use updated packages.


25/08/01 21:13:56 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1053090 ms exceeds timeout 120000 ms
25/08/01 21:13:56 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/01 21:31:42 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

: 